# Benchmarking Performance
This document contains benchmarks for the forecasting methods used. The architecture used here is compared to methods that do not include KAN layers. 

In [ ]:
#import the libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from keras_efficient_kan import KANLinear
from tqdm import tqdm

In [ ]:
# ── CONFIG ────────────────────────────────
WEATHER_CSV = "../data/clean/valais_clean.csv"
STATIONS_CSV = "../data/clean/valais_stations.csv"

HIST_LEN = 36     # 6 hours history
HORIZON = 1       # 10 minutes ahead
BATCH_SIZE = 256
EPOCHS = 10
SPLIT_FRACTION = 0.8
LEARNING_RATE = 0.0001

#set the numpy seed to make sure that the unsersampling is reproducible
np.random.seed(42)

# ── LOAD DATA ─────────────────────────────
df_weather = pd.read_csv(WEATHER_CSV)
df_weather["time"] = pd.to_datetime(df_weather["time"], format="%Y%m%d%H%M")

df_stations = pd.read_csv(STATIONS_CSV)
df = df_weather.merge(df_stations[["station", "east", "north", "altitude"]], on="station", how="left")

# ── BUILD WIDE FORMAT ─────────────────────
selected_features = ['precip', 'temperature', 'East', 'North', 'pressure', 'moisture']
metadata_features = ['east', 'north', 'altitude']

all_features = selected_features + metadata_features

df_features = df[["time", "station"] + all_features].copy()
df_pivot = df_features.pivot(index="time", columns="station", values=all_features)
df_pivot.columns = [f"{feat}_{station}" for feat, station in df_pivot.columns]
df_pivot = df_pivot.sort_index()
df_pivot = df_pivot.dropna()  


# ── SPLITTING ───────────────────────────────
split1 = int(0.6 * len(df_pivot))
split2 = int(0.8 * len(df_pivot))
df_train = df_pivot.iloc[:split1]
df_test  = df_pivot.iloc[split2:]

# ── SCALING ───────────────────────────────
scaler = StandardScaler()
data_train = scaler.fit_transform(df_train)
data_test  = scaler.transform(df_test)

# ── SAMPLE CONSTRUCTION ───────────────────
# ── SETUP ──────────────────────────────────
precip_cols = [col for col in df_pivot.columns if col.startswith("precip_")]
num_stations = len(precip_cols)

# ── TEST SET CONSTRUCTION (no undersampling) ─
x_test, y_test = [], []
test_scaled = pd.DataFrame(data_test, columns=df_test.columns, index=df_test.index)

for i in range(HIST_LEN, len(test_scaled) - HORIZON):
    x_window = test_scaled.iloc[i - HIST_LEN:i].values
    horizon_vals = df_test.iloc[i + 1 : i + 1 + HORIZON][precip_cols].values
    y_window = (np.any(horizon_vals > 0, axis=0)).astype(int)

    x_test.append(x_window)
    y_test.append(y_window)

# ── CONVERT TO ARRAYS ─────────────────────
x_test  = np.array(x_test)
y_test  = np.array(y_test)

## Binary forecasting model

In [ ]:
model_binary = keras.models.load_model('../model_testing/forecast_binary_1.keras')

## One step forecast

In [ ]:
model_wide = keras.models.load_model("../model_testing/final_one_step_fcst_wide.keras")